In [1]:
import json
import faiss
import numpy as np
import requests

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

In [2]:
with open("mne_docs_test.json", "r") as f:
    documents = json.load(f)

print(f"Loaded {len(documents)} documentation chunks.")

Loaded 60 documentation chunks.


In [6]:
embeddings = np.load("mne_embeddings.npy")

index = faiss.read_index("mne_faiss.index")

print("Embeddings shape:", embeddings.shape)
print("FAISS index loaded successfully.")

Embeddings shape: (60, 384)
FAISS index loaded successfully.


In [7]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.


In [8]:
query = """
Generate constraints and test cases
for mne.io.read_raw_edf
"""

query_embedding = embedding_model.encode([query])

query_embedding = np.array(
    query_embedding,
    dtype="float32"
)

print("Query embedding shape:", query_embedding.shape)

Query embedding shape: (1, 384)


In [9]:
k = 10

distances, indices = index.search(
    query_embedding,
    k
)

print("Retrieved indices:", indices)

Retrieved indices: [[ 0 52 55 34  5 29 58  1  3 21]]


In [10]:
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Reranker loaded successfully.")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Reranker loaded successfully.


In [11]:
retrieved_docs = []

for idx in indices[0]:
    retrieved_docs.append(documents[idx])

pairs = []

for doc in retrieved_docs:

    combined_text = f"""
    Function:
    {doc['function_name']}

    Description:
    {doc['description']}
    """

    pairs.append(
        (query, combined_text)
    )

print(f"Prepared {len(pairs)} query-document pairs.")

Prepared 10 query-document pairs.


In [12]:
scores = reranker.predict(pairs)

print(scores)

[-0.47698778 -4.6995883  -3.8610816  -7.5134726  -2.1564877  -7.2706356
 -6.365701   -0.688498   -1.2181779  -7.409832  ]


In [13]:
reranked_results = list(
    zip(
        scores,
        retrieved_docs
    )
)

reranked_results = sorted(
    reranked_results,
    key=lambda x: x[0],
    reverse=True
)

In [14]:
top_reranked_docs = reranked_results[:3]

for rank, (score, doc) in enumerate(top_reranked_docs):

    print("\n")
    print("=" * 60)

    print(f"Rank: {rank + 1}")
    print(f"Relevance Score: {score:.4f}")

    print(f"\nFunction:")
    print(doc["function_name"])

    print(f"\nDescription:")
    print(doc["description"][:500])



Rank: 1
Relevance Score: -0.4770

Function:
mne.io.read_raw_edf

Description:



Rank: 2
Relevance Score: -0.6885

Function:
mne.io.read_raw_fif

Description:



Rank: 3
Relevance Score: -1.2182

Function:
mne.io.read_raw_bdf

Description:



In [15]:
reranked_context = ""

for rank, (score, doc) in enumerate(top_reranked_docs):

    reranked_context += f"""

Rank: {rank + 1}

Relevance Score:
{score:.4f}

Function:
{doc["function_name"]}

Description:
{doc["description"]}

Parameters:
{json.dumps(doc["parameters"], indent=2)}

"""

print(reranked_context[:5000])



Rank: 1

Relevance Score:
-0.4770

Function:
mne.io.read_raw_edf

Description:


Parameters:
{
  "input_fname path-like": "Path to the EDF or EDF+ file or EDF/EDF+ file itself. If a file-like\nobject is provided, preload must be used. Changed in version 1.10: Added support for file-like objects",
  "eog list or tuple": "Names of channels or list of indices that should be designated EOG\nchannels. Values should correspond to the electrodes in the file.\nDefault is None.",
  "misc list or tuple": "Names of channels or list of indices that should be designated MISC\nchannels. Values should correspond to the electrodes in the file.\nDefault is None.",
  "stim_channel 'auto' | str | list of str | int | list of int": "Defaults to 'auto' , which means that channels named 'status' or 'trigger' (case insensitive) are set to STIM. If str (or list of\nstr), all channels matching the name(s) are set to STIM. If int (or\nlist of ints), channels corresponding to the indices are set to STIM.",
  "e

In [16]:
prompt = f"""
You are an API constraint and test generation system.

Using ONLY the reranked API documentation below,
generate parameter-level constraints and corresponding test cases.

For each inferred constraint provide:

1. Parameter Name
2. Constraint
3. Short Reasoning
4. Valid Example
5. Invalid pytest-style Test Case

Focus on:
- datatype constraints
- invalid input conditions
- mutually conflicting parameters
- filesystem-related failures
- boundary conditions

Rules:
- ONLY use behaviors supported by the documentation
- Do NOT invent undocumented parameters
- Do NOT assume hidden implementation details
- Prefer parameter-level reasoning over generic testing
- Keep outputs concise and structured
- Avoid long explanations

Reranked Documentation:
{reranked_context}
"""

In [18]:
url = "http://localhost:11434/api/generate"

payload = {
    "model": "qwen3:8b-q4_K_M",
    "prompt": prompt,
    "stream": False
}

response = requests.post(
    url,
    json=payload
)

result = response.json()

print(result["response"])

### Constraints and Test Cases for `mne.io.read_raw_edf`

1. **Parameter Name**: `input_fname`  
   **Constraint**: Must be a path-like or file-like object. If file-like, `preload` must be `True`.  
   **Reasoning**: File-like objects require preloading for data manipulation.  
   **Valid Example**: `input_fname='file.edf', preload=True`  
   **Invalid Test Case**: `input_fname=open('file.edf'), preload=False`  

2. **Parameter Name**: `eog`  
   **Constraint**: Must be a list or tuple.  
   **Reasoning**: Invalid types (e.g., strings) are not supported.  
   **Valid Example**: `eog=['EOG1', 'EOG2']`  
   **Invalid Test Case**: `eog='EOG1'`  

3. **Parameter Name**: `stim_channel`  
   **Constraint**: Must be `'auto'`, a string, list of strings, int, or list of ints.  
   **Reasoning**: Unsupported types (e.g., floats) are invalid.  
   **Valid Example**: `stim_channel='status'`  
   **Invalid Test Case**: `stim_channel=3.5`  

4. **Parameter Name**: `include`  
   **Constraint**: If p